In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy.stats import levene
from scipy.stats import spearmanr
from scipy.stats import shapiro
from scipy.stats import norm

In [3]:
# Load the dengue with weather dataset
df = pd.read_csv("dengue_data_with_weather_data.csv")
df.dropna()

# Display the first few rows to understand the structure
df.head()

df.info()

#Missing value analysis
df.isnull().sum()

#Dataframe after dropping missing values
df.dropna(inplace=True)
print('DataFrame after dropping missing values:')
df.info()

#Checking for duplicates
print(f"Number of duplicate rows: {df.duplicated().sum()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Year               900 non-null    int64  
 1   Province           900 non-null    object 
 2   District           900 non-null    object 
 3   Latitude           900 non-null    float64
 4   Longitude          900 non-null    float64
 5   Elevation          900 non-null    int64  
 6   Month              900 non-null    int64  
 7   Cases              900 non-null    int64  
 8   Temp_avg           899 non-null    float64
 9   Precipitation_avg  899 non-null    float64
 10  Humidity_avg       899 non-null    float64
dtypes: float64(5), int64(4), object(2)
memory usage: 77.5+ KB
DataFrame after dropping missing values:
<class 'pandas.core.frame.DataFrame'>
Index: 899 entries, 0 to 899
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------        

In [4]:
'''
Objective 01
Classification of geographic areas into low-, medium-, and high-risk dengue zones using dengue incidence data.
'''

#average monthly dengue cases calculated for each district 
#for the years 2019, 2020, and 2021
district_monthly_avg_cases = (
    df.groupby(['District', 'Year'])['Cases']
      .mean()
      .reset_index())


district_monthly_avg_cases.head(30)

,District,Year,Cases
0,Ampara,2019,159.000000
1,Ampara,2020,108.083333
2,Ampara,2021,35.166667
3,Anuradhapura,2019,97.000000
4,Anuradhapura,2020,36.083333
5,Anuradhapura,2021,31.000000
6,Badulla,2019,160.166667
7,Badulla,2020,43.750000
8,Badulla,2021,63.583333
9,Batticaloa,2019,237.333333


In [5]:
#Calculate the overall average dengue cases per district
# by averaging the yearly averages across the three years
district_yearly_avg_cases = (
    district_monthly_avg_cases
    .groupby('District')['Cases']
    .mean()
    .reset_index())

district_yearly_avg_cases

,District,Cases
0,Ampara,100.750000
1,Anuradhapura,54.694444
2,Badulla,89.166667
3,Batticaloa,280.000000
4,Colombo,1010.444444
5,Galle,271.111111
6,Gampaha,738.810606
7,Hambantota,80.111111
8,Jaffna,294.194444
9,Kalutara,362.250000


In [6]:
#Province
# Get the unique District to Province mapping from the original 'df' DataFrame
district_province_map = df[['District', 'Province']].drop_duplicates()

# Merge this mapping into district_year_cases to add the 'Province' column
district_year_cases_with_province = pd.merge(
    district_yearly_avg_cases,
    district_province_map,
    on='District',
    how='left'
)
district_year_cases_with_province.head()

,District,Cases,Province
0,Ampara,100.750000,Eastern
1,Anuradhapura,54.694444,North central
2,Badulla,89.166667,Uva
3,Batticaloa,280.000000,Eastern
4,Colombo,1010.444444,Western


In [7]:
# Group district averages by province to get total average cases per province
province_total_cases = (
    district_year_cases_with_province
    .groupby('Province')['Cases']
    .sum()
    .reset_index(name='total_cases')
)
province_total_cases

,Province,total_cases
0,Central,497.638889
1,Eastern,535.361111
2,North Western,264.583333
3,North central,80.694444
4,Northern,373.111111
5,Sabaragamuwa,316.055556
6,Southern,499.888889
7,Uva,103.972222
8,Western,2111.505051


In [8]:
# Define population data for each province
population = {
    'Western': 6149000,
    'Central': 2766000,
    'Southern': 2654000,
    'Northern': 1143000,
    'Eastern': 1729000,
    'North Western': 2551000,
    'North central': 1377000,
    'Uva': 1376000,
    'Sabaragamuwa': 2058000
}

In [9]:
# Map population values to each province
province_total_cases['Population'] = (
    province_total_cases['Province'].map(population)
)
province_total_cases

,Province,total_cases,Population
0,Central,497.638889,2766000
1,Eastern,535.361111,1729000
2,North Western,264.583333,2551000
3,North central,80.694444,1377000
4,Northern,373.111111,1143000
5,Sabaragamuwa,316.055556,2058000
6,Southern,499.888889,2654000
7,Uva,103.972222,1376000
8,Western,2111.505051,6149000


In [10]:
# Calculate dengue incidence per 100,000 population
province_total_cases['incidence_per_100k'] = (
    province_total_cases['total_cases'] /
    province_total_cases['Population']
) * 100000
province_total_cases

,Province,total_cases,Population,incidence_per_100k
0,Central,497.638889,2766000,17.991283
1,Eastern,535.361111,1729000,30.963627
2,North Western,264.583333,2551000,10.371750
3,North central,80.694444,1377000,5.860163
4,Northern,373.111111,1143000,32.643142
5,Sabaragamuwa,316.055556,2058000,15.357413
6,Southern,499.888889,2654000,18.835301
7,Uva,103.972222,1376000,7.556121
8,Western,2111.505051,6149000,34.338999


In [11]:
# Determine thresholds for low, medium, and high dengue risk
low_threshold = province_total_cases['incidence_per_100k'].quantile(0.33)
high_threshold = province_total_cases['incidence_per_100k'].quantile(0.66)

low_threshold, high_threshold

def classify_risk(incidence):
    if incidence <= low_threshold:
        return 'Low Risk'
    elif incidence <= high_threshold:
        return 'Medium Risk'
    else:
        return 'High Risk'

province_total_cases['risk_level'] = (
    province_total_cases['incidence_per_100k']
    .apply(classify_risk)
)

province_total_cases[
    ['Province', 'incidence_per_100k', 'risk_level']
]

,Province,incidence_per_100k,risk_level
0,Central,17.991283,Medium Risk
1,Eastern,30.963627,High Risk
2,North Western,10.371750,Low Risk
3,North central,5.860163,Low Risk
4,Northern,32.643142,High Risk
5,Sabaragamuwa,15.357413,Medium Risk
6,Southern,18.835301,Medium Risk
7,Uva,7.556121,Low Risk
8,Western,34.338999,High Risk
